This code does the following:

- defines functions to load the trajectories from the idtracker trajectories.csv and extract features + temporal features
- loads the random forest classifer ML model trained on the manual fight vs non-fight annotations

In [ ]:
import numpy as np
import pandas as pd
import os
from sklearn.linear_model import LinearRegression
import joblib

Function to extract features from idtracker trajectories.csv

In [ ]:
def features_csv(
    csv_path,
    output_csv_path,
    fps=None,               #set to fps=60, if not mentioned otherwise
    window=20,              #can be altered to test which works best
    min_bout_len=1
):
    #load idtracker trajectories and preprocess data
    
    data = pd.read_csv(csv_path)
    fish1 = data.iloc[:, [1, 2]].values
    fish2 = data.iloc[:, [3, 4]].values

    fish1 = np.nan_to_num(fish1, nan=0.0)
    fish2 = np.nan_to_num(fish2, nan=0.0)

    # Inter-animal distance
    distance = np.linalg.norm(fish1 - fish2, axis=1)

    # Speed
    if fps is None:
        fps = 1.0
    speed1 = np.linalg.norm(np.diff(fish1, axis=0, prepend=fish1[0:1]), axis=1) * fps
    speed2 = np.linalg.norm(np.diff(fish2, axis=0, prepend=fish2[0:1]), axis=1) * fps

    # Acceleration
    acc1 = np.diff(speed1, prepend=speed1[0]) * fps
    acc2 = np.diff(speed2, prepend=speed2[0]) * fps

    # Heading
    dx1 = np.diff(fish1[:, 0], prepend=fish1[0, 0])
    dy1 = np.diff(fish1[:, 1], prepend=fish1[0, 1])
    heading1 = np.arctan2(dy1, dx1)

    dx2 = np.diff(fish2[:, 0], prepend=fish2[0, 0])
    dy2 = np.diff(fish2[:, 1], prepend=fish2[0, 1])
    heading2 = np.arctan2(dy2, dx2)

    # Relative heading
    rel_heading_1tow2 = (heading2 - heading1 + np.pi) % (2 * np.pi) - np.pi
    rel_heading_2tow1 = (heading1 - heading2 + np.pi) % (2 * np.pi) - np.pi
    
    #set arbitrary fight labels based on heuristics for testing purposes

    def detect_fights(distance, speed1, speed2, window=20, min_bout_len=1): #window size can be altered to test which works best
        fight_frames = []
        for i in range(window, len(distance)):
            d_diff = distance[i - window] - distance[i]
            s1_now = speed1[i]
            s2_now = speed2[i]
            s1_diff = s1_now - speed1[i - window]
            s2_diff = s2_now - speed2[i - window]
            if (
                d_diff > 5 and
                (s1_diff > 5 or s2_diff > 5)                                #can be altered to test which works best
            ):
                fight_frames.append(i)
        bouts = []
        if not fight_frames:
            return bouts
        start = fight_frames[0]
        prev = fight_frames[0]
        for f in fight_frames[1:]:
            if f - prev > 1:
                if prev - start + 1 >= min_bout_len:
                    bouts.append((start, prev))
                start = f
            prev = f
        if prev - start + 1 >= min_bout_len:
            bouts.append((start, prev))
        return bouts

    fight_bouts = detect_fights(distance, speed1, speed2, window=window, min_bout_len=min_bout_len)

    # Frame-level fight labels
    fight_labels = np.zeros(len(distance), dtype=int)
    for start, end in fight_bouts:
        fight_labels[start:end + 1] = 1

    #define dataframe and save to csv

    df = pd.DataFrame({
        'frame': np.arange(len(distance)),
        'x1': fish1[:, 0],
        'y1': fish1[:, 1],
        'x2': fish2[:, 0],
        'y2': fish2[:, 1],
        'inter_animal_distance': distance,
        'speed1': speed1,
        'speed2': speed2,
        'acc1': acc1,
        'acc2': acc2,
        'heading1': heading1,
        'heading2': heading2,
        'relative_heading_1tow2': rel_heading_1tow2,
        'relative_heading_2tow1': rel_heading_2tow1,
        'fight_label': fight_labels
    })

    df.to_csv(output_csv_path, index=False)
    print(f"Saved features_with_labels.csv successfully to {output_csv_path}")
    return df

Function to extract temporal windowed features (temporal features can be modified, added or deleted to improve the model)

In [ ]:
def extract_temporal_features(window_df: pd.DataFrame) -> dict:
    features = {}
    window_size = len(window_df)
    t = np.arange(window_size).reshape(-1, 1)

    for col in window_df.columns:
        x = window_df[col].values
        if np.isnan(x).any():
            features[f"{col}_slope"] = np.nan
            features[f"{col}_delta"] = np.nan
            features[f"{col}_mean"] = np.nan
            features[f"{col}_std"] = np.nan
            features[f"{col}_max"] = np.nan
            features[f"{col}_min"] = np.nan
            continue
        model = LinearRegression().fit(t, x)
        features[f"{col}_slope"] = model.coef_[0]
        features[f"{col}_delta"] = x[-1] - x[0]
        features[f"{col}_mean"] = np.mean(x)
        features[f"{col}_std"] = np.std(x)
        features[f"{col}_max"] = np.max(x)
        features[f"{col}_min"] = np.min(x)

    # Chase flag
    dist_slope = features.get("inter_animal_distance_slope", 0)
    speed1_max = features.get("speed1_max", 0)
    features["chase_flag"] = int((dist_slope < -0.1) and (speed1_max > 10))

    # Burst detection
    if "speed1" in window_df.columns and not window_df["speed1"].isna().any():
        speed = window_df["speed1"].values
        speed_diff = np.diff(speed)
        features["speed1_burst_magnitude"] = np.max(speed_diff)
        features["speed1_burst_frame"] = np.argmax(speed_diff)
    else:
        features["speed1_burst_magnitude"] = np.nan
        features["speed1_burst_frame"] = -1

    # Close-then-burst logic
    if "inter_animal_distance" in window_df.columns and not window_df["inter_animal_distance"].isna().any():
        iad = window_df["inter_animal_distance"].values
        iad_min = np.min(iad)
        iad_min_frame = np.argmin(iad)
        burst_frame = features.get("speed1_burst_frame", -1)
        burst_magnitude = features.get("speed1_burst_magnitude", 0)
        burst_offset = burst_frame - iad_min_frame
        features["inter_animal_distance_min"] = iad_min
        features["inter_animal_distance_min_frame"] = iad_min_frame
        features["burst_after_contact_frames"] = burst_offset
        features["close_then_burst"] = int((iad_min < 5) and (burst_offset >= 1) and (burst_magnitude > 10))
    else:
        features["close_then_burst"] = 0
        features["burst_after_contact_frames"] = -99

    # Aggressor/defender inference
    try:
        x1 = window_df["x1"].values
        y1 = window_df["y1"].values
        x2 = window_df["x2"].values
        y2 = window_df["y2"].values

        if any(np.isnan(v).any() for v in [x1, y1, x2, y2]):
            raise ValueError("NaNs in position data")

        v1 = np.array([x1[-1] - x1[0], y1[-1] - y1[0]])
        v2 = np.array([x2[-1] - x2[0], y2[-1] - y2[0]])
        r12 = np.array([x2[-1] - x1[-1], y2[-1] - y1[-1]])
        r21 = -r12

        def safe_norm(v): return np.linalg.norm(v) + 1e-6
        cos1 = np.dot(v1, r12) / (safe_norm(v1) * safe_norm(r12))
        cos2 = np.dot(v2, r21) / (safe_norm(v2) * safe_norm(r21))

        if cos1 > cos2:
            features["aggressor_id"] = 1
            features["defender_id"] = 2
            approach_vec = r12
            aggressor_heading = window_df["heading1"].values[-1]
        else:
            features["aggressor_id"] = 2
            features["defender_id"] = 1
            approach_vec = r21
            aggressor_heading = window_df["heading2"].values[-1]

        heading_rad = np.deg2rad(aggressor_heading)
        heading_vec = np.array([np.cos(heading_rad), np.sin(heading_rad)])
        approach_vec /= safe_norm(approach_vec)
        dot = np.clip(np.dot(heading_vec, approach_vec), -1, 1)
        angle = np.arccos(dot)
        features["approach_angle_diff_deg"] = np.rad2deg(angle)

    except Exception:
        features["aggressor_id"] = -1
        features["defender_id"] = -1
        features["approach_angle_diff_deg"] = np.nan

    return features

def generate_temporal_features_from_csv(
    csv_path,
    feature_columns,
    label_column="fight_label",
    frame_column="frame",
    window_size=40,                                                              #can be altered to test which works best
):
    df = pd.read_csv(csv_path)

    # Drop rows with missing feature or label data
    df = df.dropna(subset=feature_columns + [label_column, frame_column])

    feature_rows = []
    for i in range(window_size - 1, len(df)):
        window_df = df.iloc[i - window_size + 1 : i + 1][feature_columns]

        # Skip if window contains any NaNs
        if len(window_df) != window_size or window_df.isna().any().any():
            continue

        try:
            temporal_features = extract_temporal_features(window_df)
            temporal_features["frame"] = df[frame_column].iloc[i]
            temporal_features["label"] = df[label_column].iloc[i]
            feature_rows.append(temporal_features)
        except Exception as e:
            print(f"Skipping frame {df[frame_column].iloc[i]} due to error: {e}")

    X_y_df = pd.DataFrame(feature_rows)
    return X_y_df

csv_path = "features.csv from previous function"
feature_columns = [
    'inter_animal_distance',
    'speed1', 'speed2',
    'acc1', 'acc2',
    'heading1', 'heading2',
    'relative_heading_1tow2', 'relative_heading_2tow1',
    'x1', 'y1', 'x2', 'y2'
]

X_y_df = generate_temporal_features_from_csv(
    csv_path=csv_path,
    feature_columns=feature_columns,
    label_column="fight_label",
    frame_column="frame",
    window_size=40
)

#save output
save_path = "set desired save path/features_temporal.csv"
os.makedirs(os.path.dirname(save_path), exist_ok=True)
X_y_df.to_csv(save_path, index=False)
print(f"Saved {len(X_y_df)} rows to: {save_path}")

Predict fight/non-fight labels along with probabilities for the fight labels using the trained RFC-ML model

In [ ]:
#load the trained model
clf, expected_features = joblib.load("path to trained model.pkl")

#load temporal features from previous step
df_temporal = pd.read_csv(
    "path to temporal features.csv"
)

# Convert time to frame (if needed; check if the time is in frames or seconds, it varies based on the idracker version)
df_temporal["frame"] = (df_temporal["time"] * 60).round().astype(int)  

#verify features
missing = set(expected_features) - set(df_temporal.columns)
if missing:
    raise ValueError(f"Missing expected features: {missing}")

#predict 
X_new = df_temporal[expected_features]
probs = clf.predict_proba(X_new)[:, 1]       # Probability of class 1 (fight)
preds = (probs >= 0.4).astype(int)           # Apply threshold to get predicted labels; adjust threshold if necessary (by trial and error 0.4 seemed to work best)

#save predictions with probabilities
df_temporal["predicted_label"] = preds
df_temporal["fight_probability"] = probs          

df_temporal.to_csv(
    "desired file path to save predicted labels and probabilities.csv",
    index=False
)

print("Predictions and probabilities saved based on model.")
